In [4]:
import pandas as pd

In [5]:
import boto3


In [6]:
#setting the project root to the current working directory
from pathlib import Path
import sys

current_dir = Path.cwd()

project_root = next(
    (
        path
        for path in (current_dir, *current_dir.parents)
        if (path / "dash_app" / "data.py").is_file()
    ),
    None,
)

if project_root is None:
    raise RuntimeError(
        "Could not find dash_app/data.py. Open the notebook from the project."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: c:\Users\saiou\Desktop\mapping_visualisation_project\world-population-interactive-visualisation


In [7]:
from pathlib import Path
import pandas as pd

csv_path = Path("enriched_population_data.csv")

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("Loaded the local CSV.")
else:
    import boto3
    from dash_app.data import load_population_data

    session = boto3.Session(
        profile_name="population-dev",
        region_name="eu-north-1",
    )

    s3_client = session.client("s3")

    df = load_population_data(
        bucket_name="webgis-population-app",
        object_key="data/enriched_population_data.json",
        s3_client=s3_client,
    )

    df.to_csv(csv_path, index=False)
    print("Loaded from S3 and saved a local copy.")

Loaded the local CSV.


In [8]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (14896, 7)
Columns: ['country', 'country_id', 'date', 'population', 'country_iso3', 'region', 'is_aggregate']


,country,country_id,date,population,country_iso3,region,is_aggregate
0,Africa Eastern and Southern,ZH,2025,788844284.0,AFE,Aggregates,True
1,Africa Eastern and Southern,ZH,2024,769280888.0,AFE,Aggregates,True
2,Africa Eastern and Southern,ZH,2023,750491370.0,AFE,Aggregates,True
3,Africa Eastern and Southern,ZH,2022,731821393.0,AFE,Aggregates,True
4,Africa Eastern and Southern,ZH,2021,713090928.0,AFE,Aggregates,True


In [9]:
import pandas as pd
from dash_app.data import *

# Keep only records explicitly classified as non-aggregates.
economies = df.loc[df["is_aggregate"] == False].copy()

# Convert the year to a number for comparison.
economies["date"] = pd.to_numeric(economies["date"], errors="raise")

# Find the latest year that has at least one population value.
latest_year = economies.loc[
    economies["population"].notna(), "date"
].max()

# Select economy records for that year.
latest_records = economies.loc[economies["date"] == latest_year]

economies_with_data = latest_records.loc[
    latest_records["population"].notna(), "country_id"
].nunique()

print("Latest year with population data:", latest_year)
print("Economies with a non-null population:", economies_with_data)

Latest year with population data: 2025
Economies with a non-null population: 216


In [10]:
# Simulate a user clicking France.
selected_iso3 = "FRA"

prepared_df = prepare_population_data(df)
country_history = get_country_history(prepared_df, selected_iso3)

country_history.head()

,country,country_id,date,population,country_iso3,region,is_aggregate
6607,France,FR,1970,52007169.0,FRA,Europe & Central Asia,False
6606,France,FR,1971,52499553.0,FRA,Europe & Central Asia,False
6605,France,FR,1972,52962356.0,FRA,Europe & Central Asia,False
6604,France,FR,1973,53392161.0,FRA,Europe & Central Asia,False
6603,France,FR,1974,53746571.0,FRA,Europe & Central Asia,False


In [11]:
latest_records = economies.loc[
    economies["date"] == latest_year
].copy()

with_population = latest_records.loc[
    latest_records["population"].notna()
].copy()

# Treat both missing values and empty strings as missing codes.
iso3 = with_population["country_iso3"].fillna("").str.strip()

print("Year:", latest_year)
print("Economies with population:", with_population["country_id"].nunique())
print("Distinct non-empty ISO3 codes:", iso3[iso3 != ""].nunique())
print("Records with missing ISO3 codes:", (iso3 == "").sum())

Year: 2025
Economies with population: 216
Distinct non-empty ISO3 codes: 217
Records with missing ISO3 codes: 0


In [12]:
with_population.loc[
    iso3 == "",
    ["country", "country_id", "country_iso3", "population"],
]

,country,country_id,country_iso3,population


In [13]:
#we will now compare the iso3 codes to those plotly can draw
import pandas as pd

plotly_docs_url = "https://plotly.com/python/outline-map-locations/"

# Read the tables in Plotly's documentation page.
tables = pd.read_html(plotly_docs_url)

# Find the table containing the published ISO codes.
iso_table = next(
    table for table in tables
    if "ISO Code" in table.columns
)

plotly_codes = set(
    iso_table["ISO Code"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

# Use the latest-year records with population values from our previous cell.
map_candidates = with_population.copy()
map_candidates["country_iso3"] = (
    map_candidates["country_iso3"]
    .fillna("")
    .str.strip()
    .str.upper()
)

unsupported = map_candidates.loc[
    ~map_candidates["country_iso3"].isin(plotly_codes),
    ["country", "country_id", "country_iso3"],
].drop_duplicates()

print("Economies with a population value:", map_candidates["country_id"].nunique())
print("Distinct ISO3 codes in our data:", map_candidates["country_iso3"].nunique())
print("Codes not found in Plotly's published list:", len(unsupported))

unsupported

Economies with a population value: 216
Distinct ISO3 codes in our data: 217
Codes not found in Plotly's published list: 3


,country,country_id,country_iso3
4928,Channel Islands,JG,CHI
8624,Kosovo,XK,XKX
11928,San Marino,SM,SMR


In [14]:
# Codes for economies with a population value in the selected year
data_codes = set(
    with_population["country_iso3"]
    .dropna()
    .str.strip()
    .str.upper()
)

# Codes present in our data but absent from Plotly's table
not_in_plotly = data_codes - plotly_codes

print("Unique codes in our data:", len(data_codes))
print("Codes not in Plotly's table:", not_in_plotly)

Unique codes in our data: 217
Codes not in Plotly's table: {'CHI', 'XKX', 'SMR'}


In [15]:
import plotly.express as px
import numpy as np

# Keep the notebook's existing data unchanged
map_data = with_population.copy()
map_data["country_iso3"] = (
    map_data["country_iso3"]
    .fillna("")
    .str.strip()
    .str.upper()
)
map_data['log_population'] = np.log10(map_data['population'].replace(0, np.nan))

fig = px.choropleth(
    map_data,
    locations="country_iso3",
    locationmode="ISO-3",
    color="log_population",
    color_continuous_scale="Viridis",
    hover_name="country",
    hover_data={
        "country_iso3": True,
        "population": ":,.0f",
        "date": True,
    },
    title=f"Population by economy — {latest_year}",
    labels={"population": "Population"},
)
#to make the colorbar more readable, we will set the tick values and tick text to represent population in a more understandable format
fig.update_coloraxes(
    colorbar_title="Population",
    colorbar_tickvals=[4, 5, 6, 7, 8, 9],
    colorbar_ticktext=["10K", "100K", "1M", "10M", "100M", "1B"],
)

# Give land without a matching coloured record a grey background
fig.update_geos(
    projection_type="orthographic",
    showland=True,
    landcolor="lightgrey",
)

fig.show()

In [16]:
from dash_app.ui import create_population_globe

fig = create_population_globe(with_population, latest_year)
fig.show()

In [18]:
from dash_app.ui import create_population_history_chart
import pandas as pd
import plotly.express as px

france = country_history.loc[country_history["country_iso3"] == "FRA"]
print(france.head(70))
fig = create_population_history_chart(france)
fig.show()

     country country_id  date  population country_iso3                 region  \
6607  France         FR  1970  52007169.0          FRA  Europe & Central Asia   
6606  France         FR  1971  52499553.0          FRA  Europe & Central Asia   
6605  France         FR  1972  52962356.0          FRA  Europe & Central Asia   
6604  France         FR  1973  53392161.0          FRA  Europe & Central Asia   
6603  France         FR  1974  53746571.0          FRA  Europe & Central Asia   
6602  France         FR  1975  54002853.0          FRA  Europe & Central Asia   
6601  France         FR  1976  54232383.0          FRA  Europe & Central Asia   
6600  France         FR  1977  54486467.0          FRA  Europe & Central Asia   
6599  France         FR  1978  54734030.0          FRA  Europe & Central Asia   
6598  France         FR  1979  54979851.0          FRA  Europe & Central Asia   
6597  France         FR  1980  55274184.0          FRA  Europe & Central Asia   
6596  France         FR  198

In [25]:
egypt_history = get_country_history(df, "EGY")
print(egypt_history.head(70))
fig = create_population_history_chart(egypt_history)
fig.show()

               country country_id  date   population country_iso3  \
6047  Egypt, Arab Rep.         EG  1970   34623990.0          EGY   
6046  Egypt, Arab Rep.         EG  1971   35433368.0          EGY   
6045  Egypt, Arab Rep.         EG  1972   36258328.0          EGY   
6044  Egypt, Arab Rep.         EG  1973   37102043.0          EGY   
6043  Egypt, Arab Rep.         EG  1974   37969667.0          EGY   
6042  Egypt, Arab Rep.         EG  1975   38875327.0          EGY   
6041  Egypt, Arab Rep.         EG  1976   39795557.0          EGY   
6040  Egypt, Arab Rep.         EG  1977   40748870.0          EGY   
6039  Egypt, Arab Rep.         EG  1978   41761431.0          EGY   
6038  Egypt, Arab Rep.         EG  1979   42818628.0          EGY   
6037  Egypt, Arab Rep.         EG  1980   43950413.0          EGY   
6036  Egypt, Arab Rep.         EG  1981   45147224.0          EGY   
6035  Egypt, Arab Rep.         EG  1982   46407611.0          EGY   
6034  Egypt, Arab Rep.         EG 